# 06 · NLI verifier (Prompt 12, M4), calibration (Prompt 13, M4), end-to-end table (Prompt 14, M1)

Before running:
1. **Accelerator: GPU T4 x1**. **Internet: On**.
2. **Add Input** (Your Work):
   - `fever-corpus-v1` (Datasets)
   - the outputs of notebooks **02 · BM25**, **04 · Fine-tune + hybrid** and **05 · Chain**

Expected GPU time: about 30-45 minutes. DeBERTa-v3-base scores about 300k (claim, sentence) pairs;
roberta-large-mnli runs on the 5k subset only. All logits are cached in `outputs/nli_cache/`, so Prompts 13
and 14 reuse them, and a rerun after a timeout continues where it stopped.

When it finishes: **Save Version → Save & Run All (Commit)**.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"   # public repo
BRANCH = "main"
WORK = "/kaggle/working/project"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Is Internet on in the notebook settings?")

if not os.path.exists(WORK):
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK)
else:
    git("-C", WORK, "pull", "origin", BRANCH)   # pick up teammates' latest pushes
os.chdir(WORK)
git("log", "--oneline", "-1")                    # which commit this run uses
print(os.getcwd(), sorted(os.listdir()))

In [ ]:
# Official FEVER scorer: its setup.py does not build with current pip, so we put its src/ folder on the path.
import os, subprocess
if not os.path.exists("/kaggle/working/fever-scorer"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sheffieldnlp/fever-scorer", "/kaggle/working/fever-scorer"], check=True)
os.environ["PYTHONPATH"] = "/kaggle/working/fever-scorer/src:" + os.environ.get("PYTHONPATH", "")
!pip install -q pyyaml pytest six sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Which inputs are attached? Every folder this notebook needs should appear in this list.
import glob
found = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True)
               if p.endswith(("sentences.parquet", "metrics.json", "minilm_ft/config.json", "hotpot_dev_distractor_v1.json")))
print("\n".join(found) if found else "Nothing attached: use Add Input (see the first cell)")

## Unit tests (includes the official-scorer comparison on 200 dev claims)

In [ ]:
!python -m pytest -q -rs

## Prompt 12 · NLI verifier: two models, three evidence sources, per-sentence vs concatenated premise

In [ ]:
!python -m src.verify.nli --config configs/nli.yaml

In [ ]:
from IPython.display import Markdown, Image, display
Markdown(open("outputs/verify/table.md").read())

## Prompt 13 · Temperature scaling and NEI when unsure

In [ ]:
!python -m src.verify.calibrate --config configs/calibrate.yaml

In [ ]:
import json
c = json.load(open("outputs/calibration/metrics.json"))
print(f"T = {c['T']:.3f}, tau = {c['tau']}, theta_e = {c['theta_e']}, theta_c = {c['theta_c']}")
for sp in ("val", "test"):
    s = c["splits"][sp]
    print(f"{sp}: ECE {s['ece_before']:.4f} -> {s['ece_after']:.4f} | macro-F1 {s['macro_f1_no_abstention']:.4f} -> "
          f"{s['macro_f1_with_abstention']:.4f} (coverage {s['coverage_at_tau']:.1%})")
for f in ("reliability_val.png", "reliability_test.png", "coverage.png"):
    display(Image(f"outputs/calibration/{f}", width=450))

## Prompt 14 · End-to-end pipelines, FEVER score with the official scorer, results table

One command reproduces the table.

In [ ]:
!python -m src.pipeline --table verification

In [ ]:
Markdown(open("outputs/tables/verification.md").read())

In [ ]:
!du -sh outputs/*